# PetVision AI — curated dataset v2


Run the numbered steps **in order**, using each cell's ▶ button. This notebook is self-contained: it includes the checked split manifest and training code. You do not need to upload Python scripts or install anything on your laptop.

**Before starting:** create a folder named `PetVision-AI` directly in Google Drive's **My Drive**, then upload your curated `PetVision_dataset_v2.zip` into it. Do not unzip or rename it. In Colab, select **Runtime → Change runtime type → a free GPU (T4 if offered) → Save**. No paid plan is required; free GPUs are subject to availability.

The baseline is a pretrained MobileNetV3 Small with frozen image features and a trained head per species (six cat classes, seven dog classes). Training and validation use separate groups; the final test is a separate opt-in step. This is a research baseline, not a veterinary diagnosis system. Normal-looking skin is included but does not establish whole-pet health. Softmax scores are uncalibrated model scores.

**After a disconnect:** reconnect, keep the same `RUN_NAME`, and rerun steps 1–7. Training resumes from the last fully saved epoch. An interrupted epoch will run again. Keep package versions and settings unchanged for a resume; runtime changes can require a new run.


## 1. Check the Colab runtime
Expected: `CUDA available: True` and a GPU name. If no GPU is available, stop and try later, or deliberately enable slower CPU mode below. Do not install the Windows requirements file in Colab.


In [ ]:
import sys
import torch
import torchvision
ALLOW_CPU = False #@param {type:"boolean"}
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif not ALLOW_CPU:
    raise RuntimeError("Select a GPU under Runtime > Change runtime type and rerun. Free GPU unavailable? Try later; no payment required.")
else:
    print("CPU mode selected: training will be slower.")


## 2. Connect Drive and choose the run folder
Approve Google's Drive connection prompt for your account. Keep `curated-v2` when resuming. A different name starts a new experiment, so do not change it just because the runtime disconnected.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
RUN_NAME = "curated-v2" #@param {type:"string"}
if not RUN_NAME or any(character not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_" for character in RUN_NAME):
    raise ValueError("Use letters, numbers, hyphens, and underscores for RUN_NAME.")
PROJECT_DIR = Path('/content/drive/MyDrive/PetVision-AI')
SOURCE_ZIP = PROJECT_DIR / 'PetVision_dataset_v2.zip'
RUN_DIR = PROJECT_DIR / 'runs' / RUN_NAME
if not SOURCE_ZIP.is_file():
    raise FileNotFoundError("Upload PetVision_dataset_v2.zip to My Drive/PetVision-AI first, wait for upload to finish, then rerun this cell.")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset found:", SOURCE_ZIP)
print("Checkpoints and results:", RUN_DIR)


## 3. Load the included training code and checked splits
Run this collapsed cell as-is. The split list is compressed inside the notebook; it contains filenames and hashes, not images. No code or dataset is fetched from a third-party repository.


In [ ]:
"""Dependency-free classification summaries, including duplicate-aware views."""

from collections import Counter


def classification_summary(truth, predictions, classes, weights=None):
    if not truth or len(truth) != len(predictions):
        raise ValueError("Need equally sized, nonempty labels and predictions")
    weights = [1.0] * len(truth) if weights is None else weights
    if len(weights) != len(truth) or any(weight <= 0 for weight in weights):
        raise ValueError("Weights must be positive and match observations")
    matrix = [[0.0 for _ in classes] for _ in classes]
    for actual, predicted, weight in zip(truth, predictions, weights):
        matrix[actual][predicted] += weight
    per_class = []
    for index, name in enumerate(classes):
        tp = matrix[index][index]
        support = sum(matrix[index])
        predicted_count = sum(row[index] for row in matrix)
        precision = tp / predicted_count if predicted_count else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class.append(dict(label=name, precision=precision, recall=recall, f1=f1, support=support))
    return {"accuracy": sum(matrix[i][i] for i in range(len(classes))) / sum(weights),
            "macro_f1": sum(row["f1"] for row in per_class) / len(classes),
            "per_class": per_class, "confusion_matrix": matrix,
            "matrix_order": classes, "count": len(truth)}


def metric_views(records, truth, predictions, classes):
    if len(records) != len(truth):
        raise ValueError("Metadata and prediction lengths differ")
    counts = Counter(row["group"] for row in records)
    seen, indices = set(), []
    for index, row in enumerate(records):
        if row["sha256"] not in seen:
            seen.add(row["sha256"])
            indices.append(index)
    return {"all_files": classification_summary(truth, predictions, classes),
            "unique_files": classification_summary([truth[i] for i in indices], [predictions[i] for i in indices], classes),
            "group_balanced": classification_summary(truth, predictions, classes,
                                                       [1.0 / counts[row["group"]] for row in records]),
            "independent_groups": len(counts)}

"""Small, resumable transfer-learning baseline used by the Colab notebook."""

from collections import Counter, defaultdict
import hashlib
import io
import json
from pathlib import Path
import random
import shutil
import zipfile

import torch
import torchvision
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms



PREPROCESS = {"size": 224, "mode": "fit_inside_square_then_pad", "padding_rgb": [128, 128, 128],
              "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}
CONFIG = {"architecture": "mobilenet_v3_small", "weights": "IMAGENET1K_V1", "seed": 42,
          "epochs": 20, "patience": 5, "batch_size": 16, "learning_rate": 0.001,
          "frozen_features": True, "selection": "validation_group_balanced_macro_f1",
          "preprocess": PREPROCESS}


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def manifest_hash(manifest):
    return hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()


def validate_inputs(archive, manifest):
    if sha256_file(archive) != manifest["archive_sha256"]:
        raise ValueError("Dataset ZIP differs from the audited ZIP. Upload PetVision_dataset_v2.zip.")
    included = manifest["included"]
    paths = {row["path"] for row in included}
    if len(paths) != len(included) or paths & {row.get("path") for row in manifest["excluded"]}:
        raise ValueError("Manifest has repeated or excluded paths in the included list")
    groups, hashes = defaultdict(set), defaultdict(set)
    for row in included:
        groups[row["group"]].add(row["split"])
        hashes[row["sha256"]].add(row["split"])
    if any(len(splits) > 1 for splits in list(groups.values()) + list(hashes.values())):
        raise ValueError("A duplicate group crosses splits")
    with zipfile.ZipFile(archive) as dataset:
        for row in included:
            if hashlib.sha256(dataset.read(row["path"])).hexdigest() != row["sha256"]:
                raise ValueError(f"Image hash mismatch: {row['path']}")
    for species in ("cat", "dog"):
        classes = {row["class"] for row in included if row["species"] == species}
        if classes != set(manifest["classes_by_species"][species]):
            raise ValueError("Classes differ from v2 manifest")
        for split in ("train", "validation", "test"):
            actual = {row["class"] for row in included if row["species"] == species and row["split"] == split}
            if actual != classes:
                raise ValueError(f"Missing class in {species} {split}")
    return dict(Counter(row["split"] for row in included))


def square_image(image):
    image = ImageOps.exif_transpose(image).convert("RGB")
    fitted = ImageOps.contain(image, (224, 224), method=Image.Resampling.BILINEAR)
    canvas = Image.new("RGB", (224, 224), tuple(PREPROCESS["padding_rgb"]))
    canvas.paste(fitted, ((224 - fitted.width) // 2, (224 - fitted.height) // 2))
    return canvas


class ArchiveImages(Dataset):
    def __init__(self, archive, records, classes, training=False):
        self.records = records
        self.labels = {label: index for index, label in enumerate(classes)}
        # Keep compressed image bytes in RAM; no extraction and no open ZIP handles.
        with zipfile.ZipFile(archive) as dataset:
            self.images = [dataset.read(row["path"]) for row in records]
        operations = [transforms.Lambda(square_image)]
        if training:
            operations.append(transforms.RandomHorizontalFlip())
        operations += [transforms.ToTensor(), transforms.Normalize(PREPROCESS["mean"], PREPROCESS["std"])]
        self.transform = transforms.Compose(operations)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        with Image.open(io.BytesIO(self.images[index])) as image:
            tensor = self.transform(image)
        return tensor, self.labels[self.records[index]["class"]]


def build_model(class_count, pretrained=True):
    weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.mobilenet_v3_small(weights=weights)
    for parameter in model.features.parameters():
        parameter.requires_grad = False
    model.classifier[-1] = torch.nn.Linear(model.classifier[-1].in_features, class_count)
    return model


def atomic_torch_save(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def save_json(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


def cpu_state(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def evaluate(model, dataset, classes, device, batch_size):
    model.eval()
    truth, predictions, scores = [], [], []
    total_loss = 0.0
    with torch.inference_mode():
        for images, targets in DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0):
            logits = model(images.to(device))
            total_loss += torch.nn.functional.cross_entropy(logits, targets.to(device), reduction="sum").item()
            probabilities = logits.softmax(dim=1).cpu()
            truth.extend(targets.tolist())
            predictions.extend(probabilities.argmax(dim=1).tolist())
            scores.extend(probabilities.tolist())
    metrics = metric_views(dataset.records, truth, predictions, classes)
    metrics["loss"] = total_loss / len(dataset)
    details = [{"path": row["path"], "group": row["group"], "sha256": row["sha256"],
                "actual": classes[actual], "predicted": classes[predicted], "softmax_scores": score}
               for row, actual, predicted, score in zip(dataset.records, truth, predictions, scores)]
    return metrics, details


def train_species(species, archive, manifest, run_dir, code_hash):
    if species not in {"cat", "dog"}:
        raise ValueError("Choose cat or dog")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = CONFIG["seed"] + (0 if species == "cat" else 1000)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    records = [row for row in manifest["included"] if row["species"] == species]
    classes = sorted({row["class"] for row in records})
    output = Path(run_dir) / species
    output.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output / "last.pt"
    identity = {"species": species, "classes": classes, "manifest_hash": manifest_hash(manifest),
                "config": CONFIG, "code_hash": code_hash,
                "torch": str(torch.__version__), "torchvision": str(torchvision.__version__)}
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True) if checkpoint_path.exists() else None
    if checkpoint and checkpoint["identity"] != identity:
        raise ValueError("Checkpoint settings, code, or package versions differ. Restore the prior settings or use a new RUN_NAME.")
    if (output / "test_metrics.json").exists() and (not checkpoint or not checkpoint["complete"]):
        raise ValueError("This run already has test results. Do not continue tuning this run after testing.")
    if checkpoint and checkpoint["complete"] and (output / "test_metrics.json").exists():
        print(f"{species}: this completed model has already been tested; preserving its files.")
        return json.loads((output / "validation_metrics.json").read_text())
    model = build_model(len(classes), pretrained=checkpoint is None).to(device)
    optimizer = torch.optim.Adam(model.classifier.parameters(), lr=CONFIG["learning_rate"])
    start, best, stale, best_epoch, history, best_state = 0, -1.0, 0, 0, [], None
    if checkpoint:
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        start, best, stale = checkpoint["epoch"], checkpoint["best_score"], checkpoint["stale"]
        best_epoch, history, best_state = checkpoint["best_epoch"], checkpoint["history"], checkpoint["best_state"]
        print(f"{species}: checkpoint found after epoch {start}.")
    complete = checkpoint is not None and checkpoint["complete"]
    if not complete:
        train_data = ArchiveImages(archive, [row for row in records if row["split"] == "train"], classes, training=True)
        validation_data = ArchiveImages(archive, [row for row in records if row["split"] == "validation"], classes)
        for epoch in range(start, CONFIG["epochs"]):
            # Reseeding per epoch makes a restart from the last completed epoch reproducible.
            random.seed(seed + epoch)
            torch.manual_seed(seed + epoch)
            torch.cuda.manual_seed_all(seed + epoch)
            generator = torch.Generator().manual_seed(seed + epoch)
            loader = DataLoader(train_data, batch_size=CONFIG["batch_size"], shuffle=True,
                                generator=generator, num_workers=0)
            model.eval()  # Keep frozen backbone batch-normalization statistics unchanged.
            model.classifier.train()
            running_loss = 0.0
            for images, targets in loader:
                optimizer.zero_grad(set_to_none=True)
                loss = torch.nn.functional.cross_entropy(model(images.to(device)), targets.to(device))
                if not torch.isfinite(loss):
                    raise RuntimeError("Training loss is not finite")
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * len(targets)
            metrics, _ = evaluate(model, validation_data, classes, device, CONFIG["batch_size"])
            score = metrics["group_balanced"]["macro_f1"]
            if score > best + 1e-6:
                best, stale, best_epoch, best_state = score, 0, epoch + 1, cpu_state(model)
            else:
                stale += 1
            history.append({"epoch": epoch + 1, "train_loss": running_loss / len(train_data),
                            "validation_loss": metrics["loss"],
                            "validation_accuracy": metrics["all_files"]["accuracy"],
                            "validation_group_macro_f1": score})
            complete = stale >= CONFIG["patience"] or epoch + 1 == CONFIG["epochs"]
            atomic_torch_save({"identity": identity, "epoch": epoch + 1, "model_state": cpu_state(model),
                               "optimizer_state": optimizer.state_dict(), "best_state": best_state,
                               "best_score": best, "best_epoch": best_epoch, "stale": stale,
                               "history": history, "complete": complete}, checkpoint_path)
            save_json(history, output / "history.json")
            print(f"{species} epoch {epoch+1:02d}: train loss {history[-1]['train_loss']:.3f}, "
                  f"validation accuracy {metrics['all_files']['accuracy']:.1%}, group macro-F1 {score:.3f}; saved to Drive", flush=True)
            if complete:
                break
    model.load_state_dict(best_state)
    validation_data = ArchiveImages(archive, [row for row in records if row["split"] == "validation"], classes)
    validation_metrics, _ = evaluate(model, validation_data, classes, device, CONFIG["batch_size"])
    export = {"model_state": cpu_state(model), "identity": identity,
              "best_epoch": best_epoch, "best_validation_group_macro_f1": best,
              "warning": "Research screening baseline; uncalibrated softmax scores. Normal-looking skin does not establish overall health."}
    atomic_torch_save(export, output / "best_model.pt")
    save_json(identity, output / "model_metadata.json")
    save_json(history, output / "history.json")
    save_json(validation_metrics, output / "validation_metrics.json")
    print(f"{species}: training complete. Best epoch: {best_epoch}. Saved best_model.pt.")
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return validation_metrics


def test_species(species, archive, manifest, run_dir):
    output = Path(run_dir) / species
    model_path = output / "best_model.pt"
    metrics_path = output / "test_metrics.json"
    checkpoint = torch.load(output / "last.pt", map_location="cpu", weights_only=True)
    if not checkpoint["complete"]:
        raise ValueError("Finish training before running the held-out test")
    if metrics_path.exists():
        result = json.loads(metrics_path.read_text())
        if result["model_sha256"] != sha256_file(model_path):
            raise ValueError("Tested model was changed. Do not reuse old test results.")
        print(f"{species}: returning saved test results without retesting.")
        return result
    export = torch.load(model_path, map_location="cpu", weights_only=True)
    if export["identity"]["manifest_hash"] != manifest_hash(manifest) or export["identity"]["species"] != species:
        raise ValueError("Model and dataset do not match")
    classes = export["identity"]["classes"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(len(classes), pretrained=False).to(device)
    model.load_state_dict(export["model_state"])
    records = [row for row in manifest["included"] if row["species"] == species and row["split"] == "test"]
    dataset = ArchiveImages(archive, records, classes)
    metrics, predictions = evaluate(model, dataset, classes, device, CONFIG["batch_size"])
    metrics["cohorts"] = {}
    for cohort in sorted({r["evaluation_cohort"] for r in records}):
        subset = [r for r in records if r["evaluation_cohort"] == cohort]
        cohort_metrics, _ = evaluate(model, ArchiveImages(archive, subset, classes), classes, device, CONFIG["batch_size"])
        metrics["cohorts"][cohort] = cohort_metrics
    metrics["model_sha256"] = sha256_file(model_path)
    metrics["note"] = "Contains previously exposed v1 test images. New-source patient independence is not verified. Changed taxonomy prevents direct headline comparison to v1."
    save_json(predictions, output / "test_predictions.json")
    save_json(metrics, metrics_path)
    return metrics

import base64, zlib
MANIFEST=json.loads(zlib.decompress(base64.b64decode('...')))
CODE_HASH='6c6fcad23ee6ae6a9cfa7d2c2ae5b0db355f8e7fce252c9d4d22ec7d9a7a8ab6'
print('V2 helpers and fixed splits loaded.')


## 4. Verify the ZIP and prepare this session
The notebook copies the ZIP to Colab's temporary disk for faster reading, verifies its hash, and keeps your Drive copy untouched. Expected totals: **781 train, 168 validation, 159 test**. No images are extracted. The pretrained model weights will be downloaded later by PyTorch, once per fresh runtime.


In [ ]:
LOCAL_DIR = Path('/content/petvision-session')
LOCAL_DIR.mkdir(exist_ok=True)
ARCHIVE = LOCAL_DIR / 'PetVision_dataset_v2.zip'
if not ARCHIVE.exists() or sha256_file(ARCHIVE) != MANIFEST['archive_sha256']:
    shutil.copy2(SOURCE_ZIP, ARCHIVE)
print("Verified splits:", validate_inputs(ARCHIVE, MANIFEST))
print("Excluded conflicting files:", len(MANIFEST['excluded']))
saved_manifest = RUN_DIR / 'splits.json'
if saved_manifest.exists() and manifest_hash(json.loads(saved_manifest.read_text())) != manifest_hash(MANIFEST):
    raise ValueError("This run folder belongs to different splits. Choose a new RUN_NAME.")
save_json(MANIFEST, saved_manifest)
print("Training configuration:", json.dumps(CONFIG, indent=2))


## 5. Preview training images
This shows one training image per class. These are dataset labels, not independently verified diagnoses. Test images are not shown here.


In [ ]:
import matplotlib.pyplot as plt
training_rows = [row for row in MANIFEST['included'] if row['split'] == 'train']
labels = sorted({(row['species'], row['class']) for row in training_rows})
fig, axes = plt.subplots(4, 4, figsize=(14, 13))
for axis in axes.flat: axis.axis("off")
with zipfile.ZipFile(ARCHIVE) as dataset:
    for axis, label in zip(axes.flat, labels):
        row = next(row for row in training_rows if (row['species'], row['class']) == label)
        with Image.open(io.BytesIO(dataset.read(row['path']))) as image:
            axis.imshow(square_image(image))
        axis.set_title(" / ".join(label), fontsize=9)
        axis.axis('off')
plt.tight_layout()
plt.show()


## 6. Train the cat model
Let the cell finish. Each epoch is one pass through the training images. Maximum: 20 epochs; early stopping: 5 epochs without better validation group-balanced macro-F1. The best epoch is selected using validation only. Progress is saved to Drive after every completed epoch. A first-run download of ImageNet weights is expected.


In [ ]:
cat_validation = train_species('cat', ARCHIVE, MANIFEST, RUN_DIR, CODE_HASH)


## 7. Train the dog model
This uses the same approach with dog labels and a separate checkpoint. Wait for `training complete`.


In [ ]:
dog_validation = train_species('dog', ARCHIVE, MANIFEST, RUN_DIR, CODE_HASH)


## 8. Inspect validation results
Accuracy is the fraction correctly classified. Macro-F1 gives every class equal importance. The group-balanced view gives each duplicate/candidate group total weight one. It is a sensitivity check, not proof that all cases are independent. These validation scores select the baseline; they are not final test performance.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, species in zip(axes, ('cat', 'dog')):
    history = json.loads((RUN_DIR / species / 'history.json').read_text())
    axis.plot([row['epoch'] for row in history], [row['train_loss'] for row in history], label='Training loss')
    axis.plot([row['epoch'] for row in history], [row['validation_loss'] for row in history], label='Validation loss')
    axis.set(title=species.title(), xlabel='Epoch', ylabel='Loss')
    axis.legend()
    metrics = json.loads((RUN_DIR / species / 'validation_metrics.json').read_text())
    print(species.upper(), 'validation accuracy:', round(metrics['all_files']['accuracy'], 3),
          'group-balanced macro-F1:', round(metrics['group_balanced']['macro_f1'], 3))
plt.tight_layout()
plt.show()


## 9. Evaluate the fixed test set once
After both training cells finish, check `RUN_FINAL_TEST` and run this cell. This freezes the v2 evaluation. It includes previously exposed v1 tests and a new-source holdout; it is not a fresh blind or clinical evaluation. Do not repeatedly change models based on these test results. If you rerun this cell unchanged, it reads the saved results. Small test counts and unverified labels mean these results are not clinical validation.


In [ ]:
RUN_FINAL_TEST = False #@param {type:"boolean"}
if not RUN_FINAL_TEST:
    print('Test not run. Check RUN_FINAL_TEST when you are ready to evaluate the completed baseline.')
else:
    for species in ('cat', 'dog'):
        metrics = test_species(species, ARCHIVE, MANIFEST, RUN_DIR)
        print()
        print(species.upper())
        for cohort, cohort_metrics in metrics.get('cohorts', {}).items():
            print(cohort, 'n=', cohort_metrics['all_files']['count'], 'accuracy=', cohort_metrics['all_files']['accuracy'])
        for view in ('all_files', 'unique_files', 'group_balanced'):
            report = metrics[view]
            print(f"{view}: accuracy={report['accuracy']:.1%}, macro-F1={report['macro_f1']:.3f}, images={report['count']}")
        report = metrics['unique_files']
        for row in report['per_class']:
            print(f"  {row['label']}: precision={row['precision']:.2f}, recall={row['recall']:.2f}, F1={row['f1']:.2f}, n={row['support']:.0f}")
        fig, axis = plt.subplots(figsize=(8, 6))
        axis.imshow(report['confusion_matrix'], cmap='Blues')
        names = report['matrix_order']
        axis.set_xticks(range(len(names)), names, rotation=60, ha='right')
        axis.set_yticks(range(len(names)), names)
        axis.set(xlabel='Predicted label', ylabel='Dataset label', title=species.title() + ' — unique-file test confusion matrix')
        for i, row in enumerate(report['confusion_matrix']):
            for j, value in enumerate(row):
                axis.text(j, i, str(int(value)), ha='center', va='center', color='darkred')
        plt.tight_layout()
        plt.show()


## 10. Download the models and results
Run after step 9. The download contains both best models, exact label order, preprocessing settings, validation/test metrics, predictions, histories, and split manifest. It excludes your dataset and the larger resumable checkpoints, which remain in Drive. Keep the ZIP and share the two `test_metrics.json` files with your coding assistant before building the prediction interface.


In [ ]:
from google.colab import files
required = ['best_model.pt', 'model_metadata.json', 'validation_metrics.json', 'test_metrics.json', 'test_predictions.json', 'history.json']
for species in ('cat', 'dog'):
    for name in required:
        if not (RUN_DIR / species / name).is_file():
            raise FileNotFoundError(f'Missing {species}/{name}. Finish training and step 9 first.')
download_path = LOCAL_DIR / f'PetVision_models_{RUN_NAME}.zip'
with zipfile.ZipFile(download_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(RUN_DIR / 'splits.json', 'splits.json')
    for species in ('cat', 'dog'):
        for name in required:
            bundle.write(RUN_DIR / species / name, f'{species}/{name}')
shutil.copy2(download_path, RUN_DIR / download_path.name)
print('Backup saved to Drive:', RUN_DIR / download_path.name)
files.download(str(download_path))


## Finished — or need help?

- **After downloading:** use **Runtime → Disconnect and delete runtime** to release the runtime. Your saved Drive files remain.
- **GPU unavailable:** try later, or use the explicit CPU option in step 1. Do not purchase a plan for this baseline.
- **Disconnect:** rerun steps 1–7 with the same run name; the last completed epoch is restored. Browser disconnection and runtime deletion are different; verify Drive checkpoints before assuming progress was lost.
- **Wrong ZIP/hash:** use the v2 ZIP, not a recompressed copy. Do not bypass the check.
- **Out of GPU memory:** restart the runtime first. If it repeats, share the exact error; changing batch size requires a new run to avoid mixing configurations.
- **Changed library version on resume:** the notebook stops rather than mixing versions. Share the error; do not delete existing checkpoints.
- **Low scores:** send the metrics and learning curves. We will investigate label quality and generalization; there is no promised accuracy target.
- **A normal or unrelated image:** the normal-looking skin class does not establish health, and this model cannot recognize all unsupported inputs. Do not interpret its highest score as a diagnosis.

References: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [PyTorch MobileNetV3 Small](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.mobilenet_v3_small.html).
